In [1]:
import pandas as pd
import numpy as np

# ==========================================
# ส่วนที่ 1: การตั้งค่าข้อมูล (Configuration)
# แก้ไขข้อมูลตรงนี้เพื่อ เพิ่ม/ลด สินค้า หรือ เดือน
# ==========================================

# 1.1 ข้อมูลสินค้า (Product Specs)
# เพิ่มสินค้าใหม่โดยใส่ Key เพิ่มใน Dict นี้
product_data = {
    'A': {'I0': 50,  'R': 8,  'Alloc_Pct': 12.5},
    'B': {'I0': 100, 'R': 10, 'Alloc_Pct': 25.0},
    'C': {'I0': 200, 'R': 12, 'Alloc_Pct': 50.0},
    'D': {'I0': 50,  'R': 6,  'Alloc_Pct': 12.5},
    # ตัวอย่างการเพิ่มสินค้า: 'E': {'I0': 0, 'R': 5, 'Alloc_Pct': 0},
}

# 1.2 ข้อมูลความต้องการลูกค้า (Forecast Demand)
# เพิ่มเดือนใหม่โดยเพิ่ม Key เดือน และใส่ยอดขายสินค้าให้ครบ
demand_data = {
    'ม.ค.': {'A': 400, 'B': 520, 'C': 600, 'D': 400},
    'ก.พ.': {'A': 350, 'B': 420, 'C': 500, 'D': 335},
    'มี.ค.': {'A': 260, 'B': 310, 'C': 360, 'D': 250},
    # ตัวอย่างการเพิ่มเดือน: 'เม.ย.': {'A': 300, 'B': 400, 'C': 400, 'D': 300},
}

# 1.3 ข้อมูลแผนรวม (Aggregate Plan)
# ต้องมีเดือนตรงกับ demand_data
aggregate_plan_data = {
    'ม.ค.': {'Target_Inv_Kg': 4050, 'Max_RM_Limit': 18500},
    'ก.พ.': {'Target_Inv_Kg': 3375, 'Max_RM_Limit': 18750},
    'มี.ค.': {'Target_Inv_Kg': 2475, 'Max_RM_Limit': 10000},
    # ตัวอย่างการเพิ่มเดือน: 'เม.ย.': {'Target_Inv_Kg': 2000, 'Max_RM_Limit': 15000},
}

# ==========================================
# ส่วนที่ 2: ระบบคำนวณอัตโนมัติ (Processing Logic)
# (ไม่ต้องแก้ไขส่วนนี้ ระบบจะรันตามข้อมูลข้างบน)
# ==========================================

# แปลงข้อมูลเป็น DataFrame เพื่อความง่ายในการคำนวณ
df_products = pd.DataFrame(product_data).T  # Index = Product Name
df_demand = pd.DataFrame(demand_data)       # Index = Product, Col = Month
df_agg = pd.DataFrame(aggregate_plan_data).T # Index = Month

products_list = df_products.index.tolist()
months_list = df_demand.columns.tolist()

# เตรียมตัวแปรเก็บผลลัพธ์
results = {
    'I': pd.DataFrame(index=products_list, columns=months_list),
    'D': pd.DataFrame(index=products_list, columns=months_list),
    'd': pd.DataFrame(index=products_list, columns=months_list)
}

# เริ่มต้น Inventory จาก I0 ของแต่ละสินค้า
current_inv = df_products['I0'].copy()

# ลูปคำนวณทีละเดือน
for month in months_list:
    # 1. ดึงค่าเป้าหมาย Inventory รวม (kg) ของเดือนนั้น
    target_kg = df_agg.loc[month, 'Target_Inv_Kg']
    
    # 2. คำนวณ I (Inventory ปลายงวดหน่วยสินค้า)
    # สูตร: (Target_Kg * %Alloc) / (100 * R)
    inv_t = (target_kg * (df_products['Alloc_Pct'] / 100)) / df_products['R']
    inv_t = inv_t.round().astype(int) # ปัดเศษตามโจทย์
    results['I'][month] = inv_t
    
    # 3. คำนวณ D (Net Requirement)
    # สูตร: FD - I_prev
    fd_t = df_demand[month]
    d_req_t = fd_t - current_inv
    results['D'][month] = d_req_t
    
    # 4. คำนวณ d (Production Plan)
    # สูตร Correction: d = D + I_current
    d_plan_t = d_req_t + inv_t
    results['d'][month] = d_plan_t
    
    # อัปเดต Inventory ต้นงวดรอบหน้า ให้เป็นปลายงวดรอบนี้
    current_inv = inv_t

# ==========================================
# ส่วนที่ 3: จัดรูปแบบตารางแสดงผล (Display Formatting)
# ==========================================

# สร้าง DataFrame รวมสำหรับแสดงผล
final_df = df_products[['I0', 'R']].copy()
final_df.columns = ['I0', 'Raw mat/unit'] # Rename เพื่อความสวยงาม

# รวมข้อมูล FD, D, d, I เข้าไป
for m in months_list: final_df[f'FD_{m}'] = df_demand[m]
for m in months_list: final_df[f'D_{m}'] = results['D'][m]
for m in months_list: final_df[f'd_{m}'] = results['d'][m]

final_df['%_Alloc'] = df_products['Alloc_Pct']
for m in months_list: final_df[f'I_{m}'] = results['I'][m]

# จัด Group Header (MultiIndex)
cols = [('Basic Info', 'I0'), ('Basic Info', 'Raw mat/unit')]
cols += [('ความต้องการ (FD)', m) for m in months_list]
cols += [('ต้องการผลิต (D)', m) for m in months_list]
cols += [('แผนการผลิตจริง (d)', m) for m in months_list]
cols += [('Inventory (I)', '%kg')]
cols += [('Inventory (I)', m) for m in months_list]

final_df.columns = pd.MultiIndex.from_tuples(cols)

# ==========================================
# ส่วนที่ 4: สรุปและตรวจสอบทรัพยากร (Summary Check)
# ==========================================

summary_rows = []
status_rows = []

for month in months_list:
    # คำนวณการใช้วัตถุดิบจริง: Sum(d * R)
    actual_d = results['d'][month]
    rm_per_unit = df_products['R']
    total_usage = (actual_d * rm_per_unit).sum()
    
    limit = df_agg.loc[month, 'Max_RM_Limit']
    
    summary_rows.append(total_usage)
    
    if total_usage > limit:
        diff = total_usage - limit
        status_rows.append(f"❌ เกิน (+{int(diff)})")
    else:
        status_rows.append("✅ ปกติ")

summary_df = pd.DataFrame({
    'แผนย่อย Heuristic (kg)': summary_rows,
    'แผนรวม Aggregate Limit (kg)': df_agg['Max_RM_Limit'].values,
    'สถานะ': status_rows
}, index=months_list).T

# แสดงผล
print(f"=== ผลลัพธ์การคำนวณ ({len(products_list)} Products x {len(months_list)} Months) ===")
display(final_df)
print("\n=== การตรวจสอบวัตถุดิบ (Resource Check) ===")
display(summary_df)

=== ผลลัพธ์การคำนวณ (4 Products x 3 Months) ===


Basic Info              ความต้องการ (FD)            ต้องการผลิต (D)       \
          I0 Raw mat/unit             ม.ค. ก.พ. มี.ค.            ม.ค. ก.พ.   
A       50.0          8.0              400  350   260           350.0  287   
B      100.0         10.0              520  420   310           420.0  319   
C      200.0         12.0              600  500   360           400.0  331   
D       50.0          6.0              400  335   250           350.0  251   

        แผนการผลิตจริง (d)            Inventory (I)                  
  มี.ค.               ม.ค. ก.พ. มี.ค.           %kg ม.ค. ก.พ. มี.ค.  
A   207              413.0  340   246          12.5   63   53    39  
B   226              521.0  403   288          25.0  101   84    62  
C   219              569.0  472   322          50.0  169  141   103  
D   180              434.0  321   232          12.5   84   70    52


=== การตรวจสอบวัตถุดิบ (Resource Check) ===


,ม.ค.,ก.พ.,มี.ค.
แผนย่อย Heuristic (kg),17946.0,14340.0,10104.0
แผนรวม Aggregate Limit (kg),18500,18750,10000
สถานะ,✅ ปกติ,✅ ปกติ,❌ เกิน (+104)
